In [1]:
import numpy as np 
import pandas as pd 
import warnings
warnings.filterwarnings("ignore")
import os
pd.plotting.register_matplotlib_converters()
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set_style("dark")
sns.set_palette("viridis")
from sklearn.preprocessing import  LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, recall_score, precision_score
import optuna


<div id="1" style="background-color: #1a1a1a; padding: 10px; border-radius: 10px; border: 2px solid black;">
    <h1 style="font-family:  'Garamond', 'Lucida Sans', sans-serif; text-align: center; color: #fff; font-weight: bold; font-size: 42px;">
    Dataset Overview
    </h1>
    <a class="anchor"  id="chapter1"></a>
</div>

In [2]:
train_data=pd.read_csv("../data/train.csv")


In [3]:
# Split into train and test (so test_data exists for preprocessing below)
train_data, test_data = train_test_split(train_data, test_size=0.2, random_state=42, stratify=train_data["Exited"])

In [4]:
# Binary flags: ensure int (0/1) not float
for col in ("HasCrCard", "IsActiveMember"):
    if col in train_data.columns:
        train_data[col] = train_data[col].astype(int)
    if col in test_data.columns:
        test_data[col] = test_data[col].astype(int)

In [5]:
train_data.Exited

112149    0
70095     0
29247     0
161355    0
105992    0
         ..
123771    1
22900     0
40851     1
52537     0
75738     0
Name: Exited, Length: 132027, dtype: int64

In [6]:
train_indices = train_data[train_data.Exited.notnull()].index
train_data.loc[train_indices, "Exited"]  

112149    0
70095     0
29247     0
161355    0
105992    0
         ..
123771    1
22900     0
40851     1
52537     0
75738     0
Name: Exited, Length: 132027, dtype: int64

In [7]:
train_data.head()

,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
112149,112149,15809838,Genovese,726,France,Female,35.0,1,0.00,2,1,1,141466.85,0
70095,70095,15815645,Akhtar,481,France,Female,37.0,8,152303.66,2,1,0,175082.20,0
29247,29247,15728005,P'eng,583,France,Female,35.0,5,0.00,1,0,0,102581.11,0
161355,161355,15641136,Davison,644,France,Female,32.0,7,0.00,2,1,0,77965.67,0
105992,105992,15681671,Chibueze,709,France,Male,29.0,5,128548.49,1,1,1,140941.47,0


In [8]:
test_data.head()

,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
47555,47555,15656901,Hs?,590,France,Male,44.0,8,0.00,2,1,0,87067.73,0
142335,142335,15750264,Palermo,804,France,Male,34.0,1,0.00,3,1,0,116495.55,1
125861,125861,15709917,H?,579,France,Male,32.0,9,0.00,2,1,0,161253.08,0
13099,13099,15674851,T'ien,622,France,Male,38.0,3,0.00,1,0,0,105295.77,0
96966,96966,15663410,Genovesi,652,France,Male,29.0,9,125552.96,1,0,0,181605.85,0


In [9]:
print("train shape: ", train_data.shape, "test shape: ", test_data.shape)

train shape:  (132027, 14) test shape:  (33007, 14)


In [10]:
train_data.describe()

,id,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,132027.000000,1.320270e+05,132027.000000,132027.000000,132027.000000,132027.000000,132027.000000,132027.000000,132027.000000,132027.000000,132027.000000
mean,82553.321071,1.569204e+07,656.497898,38.119756,5.017072,55457.889422,1.553773,0.754785,0.496959,112485.647550,0.211601
std,47601.337874,7.145115e+04,80.072181,8.860025,2.804446,62791.158845,0.547302,0.430216,0.499993,50308.143749,0.408445
min,0.000000,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.000000,0.000000,11.580000,0.000000
25%,41399.500000,1.563306e+07,597.000000,32.000000,3.000000,0.000000,1.000000,1.000000,0.000000,74588.410000,0.000000
50%,82475.000000,1.569016e+07,659.000000,37.000000,5.000000,0.000000,2.000000,1.000000,0.000000,117633.960000,0.000000
75%,123834.500000,1.575689e+07,710.000000,42.000000,7.000000,119859.555000,2.000000,1.000000,1.000000,155071.560000,0.000000
max,165033.000000,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.000000,1.000000,199992.480000,1.000000


In [11]:
train_data.Exited.value_counts()

Exited
0    104090
1     27937
Name: count, dtype: int64

<div id="3" style="background-color: #1a1a1a; padding: 10px; border-radius: 10px; border: 2px solid black;">
    <h1 style="font-family:  'Garamond', 'Lucida Sans', sans-serif; text-align: center; color: #fff; font-weight: bold; font-size: 42px;">
    Preprocessing the Data
    </h1>
    <a class="anchor"  id="chapter3"></a>
</div>

In [12]:
# One encoder per column (use 1D series for LabelEncoder)
le_gender = LabelEncoder()
le_geography = LabelEncoder()

train_data["Gender"] = le_gender.fit_transform(train_data["Gender"].astype(str))
test_data["Gender"] = le_gender.transform(test_data["Gender"].astype(str))

train_data["Geography"] = le_geography.fit_transform(train_data["Geography"].astype(str))
test_data["Geography"] = le_geography.transform(test_data["Geography"].astype(str))

# Hash Surname to handle high cardinality and unseen values in test
import hashlib
def hash_surname(s):
    return int(hashlib.md5(str(s).encode()).hexdigest()[:8], 16) % 1000

train_data["Surname"] = train_data["Surname"].apply(hash_surname)
test_data["Surname"] = test_data["Surname"].apply(hash_surname)

train_data.head()

,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
112149,112149,15809838,914,726,0,0,35.0,1,0.00,2,1,1,141466.85,0
70095,70095,15815645,756,481,0,0,37.0,8,152303.66,2,1,0,175082.20,0
29247,29247,15728005,180,583,0,0,35.0,5,0.00,1,0,0,102581.11,0
161355,161355,15641136,595,644,0,0,32.0,7,0.00,2,1,0,77965.67,0
105992,105992,15681671,255,709,0,1,29.0,5,128548.49,1,1,1,140941.47,0


<div id="4" style="background-color: #1a1a1a; padding: 10px; border-radius: 10px; border: 2px solid black;">
    <h1 style="font-family:  'Garamond', 'Lucida Sans', sans-serif; text-align: center; color: #fff; font-weight: bold; font-size: 42px;">
    Models
    </h1>
    <a class="anchor"  id="chapter4
                           "></a>
</div>

In [13]:
# Features: buckets, interactions, geo frequency 
df_train = train_data.copy()
df_test = test_data.copy()

# 1. Ratio and membership features
df_train['Mem__no__Products'] = df_train['NumOfProducts'] * df_train['IsActiveMember']
df_test['Mem__no__Products'] = df_test['NumOfProducts'] * df_test['IsActiveMember']
df_train['Balance_Salary_Ratio'] = np.where(df_train['EstimatedSalary'] > 0, df_train['Balance'] / df_train['EstimatedSalary'], 0.0)
df_test['Balance_Salary_Ratio'] = np.where(df_test['EstimatedSalary'] > 0, df_test['Balance'] / df_test['EstimatedSalary'], 0.0)
df_train['Balance_Age_Ratio'] = np.where(df_train['Age'] > 0, df_train['Balance'] / df_train['Age'], 0.0)
df_test['Balance_Age_Ratio'] = np.where(df_test['Age'] > 0, df_test['Balance'] / df_test['Age'], 0.0)

# 2. Age and Tenure buckets
age_bins = [0, 25, 35, 45, 55, 120]
age_labels = [0, 1, 2, 3, 4]
df_train["AgeBucket"] = pd.cut(df_train["Age"], bins=age_bins, labels=age_labels, include_lowest=True).astype(int)
df_test["AgeBucket"] = pd.cut(df_test["Age"], bins=age_bins, labels=age_labels, include_lowest=True).astype(int)

tenure_bins = [0, 2, 5, 10, 20]
tenure_labels = [0, 1, 2, 3]
df_train["TenureBucket"] = pd.cut(df_train["Tenure"], bins=tenure_bins, labels=tenure_labels, include_lowest=True).astype(int)
df_test["TenureBucket"] = pd.cut(df_test["Tenure"], bins=tenure_bins, labels=tenure_labels, include_lowest=True).astype(int)

# 3. Interaction features
df_train["Age_Tenure"] = df_train["Age"] * df_train["Tenure"]
df_test["Age_Tenure"] = df_test["Age"] * df_test["Tenure"]
df_train["CreditScore_IsActive"] = df_train["CreditScore"] * df_train["IsActiveMember"]
df_test["CreditScore_IsActive"] = df_test["CreditScore"] * df_test["IsActiveMember"]
df_train["Tenure_NumProducts"] = df_train["Tenure"] * df_train["NumOfProducts"]
df_test["Tenure_NumProducts"] = df_test["Tenure"] * df_test["NumOfProducts"]

# 4. Geography frequency encoding
geo_freq = df_train["Geography"].value_counts(normalize=True)
df_train["Geography_freq"] = df_train["Geography"].map(geo_freq)
df_test["Geography_freq"] = df_test["Geography"].map(geo_freq).fillna(0)

# 5. Build feature matrices (drop ids and target)
drop_cols = ["Exited", "CustomerId"]
if "id" in df_train.columns:
    drop_cols.append("id")
X_train = df_train.drop(columns=drop_cols)
y_train = df_train["Exited"]
X_test = df_test[X_train.columns]
y_test = df_test["Exited"]

In [14]:
# Shared train/validation split for Optuna
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train,
)

In [15]:
test_data.head()

,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
47555,47555,15656901,733,590,0,1,44.0,8,0.00,2,1,0,87067.73,0
142335,142335,15750264,507,804,0,1,34.0,1,0.00,3,1,0,116495.55,1
125861,125861,15709917,465,579,0,1,32.0,9,0.00,2,1,0,161253.08,0
13099,13099,15674851,517,622,0,1,38.0,3,0.00,1,0,0,105295.77,0
96966,96966,15663410,300,652,0,1,29.0,9,125552.96,1,0,0,181605.85,0


## Optuna + XGBoost

Single Optuna study; best trial selected by recall (ROC-AUC tie-breaker). Retrain on full training data and evaluate on test.

In [16]:
# Optuna hyperparameter optimization for XGBClassifier
# Primary metric: recall; ROC-AUC tie-breaker applied after study.
# Uses shared X_tr, X_val, y_tr, y_val from previous cell.

def _best_trial_by_recall_then_roc_auc(study, tolerance=1e-4):
    """Select trial with max validation recall; use ROC-AUC as tie-breaker."""
    trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    if not trials:
        return None
    best = max(trials, key=lambda t: (t.user_attrs.get("recall", 0), t.user_attrs.get("roc_auc", 0)))
    return best

# Scale positive class weight for imbalance (churn = minority)
scale_pos_weight = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)


def objective_xgb(trial: optuna.Trial) -> float:
    """Optuna objective: maximize recall, use ROC-AUC as tie-breaker."""
    params = {
        "use_label_encoder": False,
        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1,
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "scale_pos_weight": scale_pos_weight,
    }
    model = XGBClassifier(**params)
    model.fit(X_tr, y_tr)
    y_val_proba = model.predict_proba(X_val)[:, 1]
    y_val_pred = (y_val_proba >= 0.5).astype(int)
    recall = recall_score(y_val, y_val_pred, zero_division=0)
    f1 = f1_score(y_val, y_val_pred, zero_division=0)
    roc_auc = roc_auc_score(y_val, y_val_proba)
    trial.set_user_attr("recall", recall)
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("roc_auc", roc_auc)
    return recall


study_xgb = optuna.create_study(direction="maximize", study_name="xgb_churn_study")
study_xgb.optimize(objective_xgb, n_trials=30, n_jobs=1)

# Primary metric: recall; ROC-AUC used only to break ties
best_trial_xgb = _best_trial_by_recall_then_roc_auc(study_xgb)
print("Best trial (recall primary, ROC-AUC tie-breaker):")
print(f"  Validation recall: {best_trial_xgb.user_attrs.get('recall', 0):.4f}")
print(f"  Validation F1:     {best_trial_xgb.user_attrs.get('f1', 0):.4f}")
print(f"  Validation ROC-AUC: {best_trial_xgb.user_attrs.get('roc_auc', 0):.4f}")
print("Best hyperparameters:")
for k, v in best_trial_xgb.params.items():
    print(f"  {k}: {v}")

best_params_xgb = best_trial_xgb.params.copy()
best_params_xgb.update({
    "use_label_encoder": False,
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1,
    "scale_pos_weight": scale_pos_weight,
})

xgb_optuna = XGBClassifier(**best_params_xgb)
xgb_optuna.fit(X_train, y_train)

y_test_proba = xgb_optuna.predict_proba(X_test)[:, 1]
y_test_pred = xgb_optuna.predict(X_test)
roc_auc = roc_auc_score(y_test, y_test_proba)
acc = accuracy_score(y_test, y_test_pred)
print(f"Optuna XGB ROC-AUC on test: {roc_auc:.4f}")
print(f"Optuna XGB accuracy on test: {acc:.4f}")


[I 2026-03-07 17:29:16,379] A new study created in memory with name: xgb_churn_study
[I 2026-03-07 17:29:20,167] Trial 0 finished with value: 0.6932712956335003 and parameters: {'n_estimators': 362, 'learning_rate': 0.06246882887523113, 'max_depth': 10, 'min_child_weight': 1, 'subsample': 0.9608059501816256, 'colsample_bytree': 0.8977777296613563, 'reg_alpha': 1.933101502410057e-08, 'reg_lambda': 1.6966453552380064e-06}. Best is trial 0 with value: 0.6932712956335003.
[I 2026-03-07 17:29:24,260] Trial 1 finished with value: 0.6871868289191124 and parameters: {'n_estimators': 687, 'learning_rate': 0.09300495721733622, 'max_depth': 9, 'min_child_weight': 4, 'subsample': 0.9347839325987455, 'colsample_bytree': 0.6748801896981846, 'reg_alpha': 0.0026414939731174733, 'reg_lambda': 5.011394021553296e-06}. Best is trial 0 with value: 0.6932712956335003.
[I 2026-03-07 17:29:31,685] Trial 2 finished with value: 0.711882605583393 and parameters: {'n_estimators': 645, 'learning_rate': 0.025937964

Best trial (recall primary, ROC-AUC tie-breaker):
  Validation recall: 0.8071
  Validation F1:     0.6380
  Validation ROC-AUC: 0.8876
Best hyperparameters:
  n_estimators: 333
  learning_rate: 0.02336215651997043
  max_depth: 4
  min_child_weight: 9
  subsample: 0.8174785656036242
  colsample_bytree: 0.7780044907402758
  reg_alpha: 0.2677873672182936
  reg_lambda: 0.0006881967373572972
Optuna XGB ROC-AUC on test: 0.8895
Optuna XGB accuracy on test: 0.8124


In [17]:
# Retrain best XGB on full training data and evaluate on test set
scale_pos_weight_full = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
best_params = best_trial_xgb.params.copy()
best_params.update({
    "use_label_encoder": False,
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1,
    "scale_pos_weight": scale_pos_weight_full,
})
best_model = XGBClassifier(**best_params)
best_model.fit(X_train, y_train)

y_test_proba = best_model.predict_proba(X_test)[:, 1]
y_test_pred = best_model.predict(X_test)
roc_auc = roc_auc_score(y_test, y_test_proba)
acc = accuracy_score(y_test, y_test_pred)
prec = precision_score(y_test, y_test_pred)
rec = recall_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
print("Best XGB on test:")
print(f"  ROC-AUC:   {roc_auc:.4f}")
print(f"  Accuracy:  {acc:.4f}")
print(f"  Recall:    {rec:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  F1:        {f1:.4f}")

Best XGB on test:
  ROC-AUC:   0.8895
  Accuracy:  0.8124
  Recall:    0.7975
  Precision: 0.5383
  F1:        0.6428
